# A support handover with Super VL and automatic routing

One agentic task: assemble a factual handover from a **79-message historical Ubuntu support conversation**. The agent retrieves the environment, contradictory upgrade history, remediation and confirmation, then submits a structured report.

**Efficient executor and judge:** `nvidia/nvidia/nemotron-3.5-super-vl`  
**Capable executor:** the existing GPT Sol model. The original synthetic and storage demos remain available as workshop backups.

In [ ]:
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_workshop = next(p for p in [_start, *_start.parents] if (p / 'shared/incident_demo.py').is_file())
sys.path.insert(0, str(_workshop / 'shared'))


In [ ]:
from pathlib import Path
from copy import deepcopy
import json, os, html
from getpass import getpass
from dotenv import load_dotenv
from IPython.display import display, HTML
import incident_demo as core
import ubuntu_handover_demo as handover
import ubuntu_savings_demo as savings
ROOT = core.ROOT
load_dotenv(ROOT / '.env', override=True)


## Choose a recording or run a fresh comparison

Recorded mode uses saved real model responses and makes no API calls. Live mode runs three attempts per approach. All failures and judge calls count toward the comparison.

In [ ]:
DEMO_MODE = "recorded"  # recorded | live
ALLOW_PAID_CALLS = False
REPEATS = 3
REPLAY_PATH = ROOT / 'recordings/ubuntu-handover/super-vl-both-v2-2026-10-16-20261016T042519610347Z.json'  # verified complete recording
PROFILE = ROOT / 'configs/model-profiles/nemotron-super-vl-gpt-sol.toml'
STRATEGY = ROOT / 'configs/routing/task-classifier.toml'
assert DEMO_MODE in {'recorded', 'live'}
settings = core.load_settings(PROFILE, STRATEGY, savings.PRICING,
    mode='live' if DEMO_MODE == 'live' else 'rehearsal', allow_paid_calls=ALLOW_PAID_CALLS)
settings.enable_langsmith = False
if DEMO_MODE == 'live':
    for endpoint in (settings.efficient, settings.frontier, settings.judge):
        if not os.getenv(endpoint.key_env):
            os.environ[endpoint.key_env] = getpass(f'{endpoint.key_env}: ')
    settings.validate()
case = handover.load_case()
print(case['summary'])
print('Evidence sections:', ', '.join(case['evidence']))
print(case['missing_evidence'])


In [ ]:
if DEMO_MODE == 'recorded':
    paths = [Path(REPLAY_PATH)] if REPLAY_PATH else sorted(
        (ROOT / 'recordings/ubuntu-handover').glob('super-vl-both-v2-2026-10-16-*.json'), reverse=True)
    bundle = None
    for path in paths:
        candidate = json.loads(path.read_text())
        rows = candidate.get('results', [])
        pairs = {(r['arm'], r.get('repeat')) for r in rows}
        if len(rows) == 9 and pairs == {(a,r) for a in core.ARMS for r in (1,2,3)}:
            bundle = candidate
            break
    if bundle is None:
        raise RuntimeError('No complete matched recording found. Run the live comparison first.')
    results = bundle['results']
    for role in ('efficient', 'frontier', 'judge'):
        getattr(settings, role).model = bundle['metadata']['settings'][role]['model']
    scenario_prices = deepcopy(bundle['metadata']['scenario_prices'])
    print('RECORDED LIVE RESPONSES:', path.name)
    print('Interface:', bundle['metadata']['interface_revision'])
else:
    results, path = await handover.compare(settings, repeats=REPEATS, phase='super-vl-workshop')
    scenario_prices = savings.prices(settings)
    print('Saved:', path.name)


## Costs, passes and routing overhead

Super VL uses **$0.25/M input and $0.65/M output**, requested from Artificial Analysis. Cache reads retain the user's **$0.10/M** assumption; cache writes are assumed to cost ordinary input at **$0.25/M**. The capable model retains the earlier illustrative rates. These are comparison prices, not an invoice from the gateway.

The $0.30 Intelligence Index average task cost is **not** used as a fee here. We price this task's actual returned input, output and cache usage, including every judge call and failed attempt.

In [ ]:
# Edit scenario_prices and rerun this cell to reprice saved usage without inference.
priced_results = core.price_demo_results(results, scenario_prices)
summary = core.summarize(priced_results)
rows = []
for _, item in summary.iterrows():
    rows.append({
        'Approach': {'Efficient only':'Super VL only', 'Switchyard':'Automatic routing'}.get(item.arm, item.arm),
        'Checks passed': f"{int(item.passed)}/{int(item.tasks)}",
        'Mean USD / attempt': f"${item.mean_cost_usd:.5f}" if item.cost_complete else 'unavailable',
        'USD / accepted report': f"${item.cost_per_pass_usd:.5f}" if core.pd.notna(item.cost_per_pass_usd) else 'no passes',
        'Savings vs capable': f"{item.savings_vs_frontier:.1%}" if core.pd.notna(item.savings_vs_frontier) else 'unavailable',
        'Median seconds': round(item.median_seconds, 1),
    })
display(core.pd.DataFrame(rows))
print('Costs include every judge call and failed attempt. Passes are automated checks; inspect report wording too.')
print('Super-VL-only has lower attempt cost. This small example does not establish a general reliability advantage for routing.')


## Inspect a recorded model switch

The report must identify the admitted interrupted developer upgrade, the reported successful remedy and the missing post-reboot kernel observation. Historical helper interpretations need attribution. Report text fields accept prose or lists and are stored as lists; this does not change the factual checks.

The default selection is repeat 2: capable reads the ticket, then Super VL retrieves evidence and writes the report. The other two routed attempts use Super VL throughout. The comparison above includes all three; this trace does not show capable-only reasoning on a difficult step.

In [ ]:
routed = sorted([r for r in priced_results if r['arm'] == 'switchyard'], key=lambda r:r['repeat'])
SELECTED_REPEAT = 2  # 1, 2 or 3; repeat 2 shows a recorded model switch
r = next(item for item in routed if item['repeat'] == SELECTED_REPEAT)
print('Selected routed attempt:', r['repeat'], 'automated checks:', r['passed'])
display(HTML('<pre style="white-space:pre-wrap">' + html.escape(json.dumps(r['report'], indent=2)) + '</pre>'))
display(core.pd.DataFrame(r['calls']).reindex(columns=[
    'call','role','model','input_tokens','output_tokens','cost_usd',
    'classifier_p_solve','classifier_boundary','classifier_verdict_valid']))
display(core.pd.DataFrame([{'Check':k,'Pass':v} for k,v in r['checks'].items()]))
if r.get('submission_diagnostics'):
    print(r['submission_diagnostics'])


## Interpretation and source

Compare routing with both capable-only and Super-VL-only. If Super-VL-only solves this task for less, show that result. One selected conversation and three repetitions do not establish general production reliability or savings.

This task summarizes a **completed** conversation, so its historical remedy and outcome are available to the agent. It performs no repair and retrieves no missing paste contents. Earlier experiments with strict list-only report fields remain saved separately; they are not pooled into this interface's results.

Source: [Ubuntu Chat Logs Corpus](https://www.convokit.cornell.edu/documentation/chatlogs.html), conversation 79.2; Sarkar et al. (2025), ConvoKit formatting by Axel Bax, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). All 79 source messages are preserved under `data/ubuntu/handover/`. Grouping, tools and evaluation are our adaptation.

Pricing reference: [Artificial Analysis](https://artificialanalysis.ai/models/nemotron-3-5-super-vl). Original workshop fallback: `examples/synthetic-incident/demo.ipynb`.